### Data Loading and Label Categorisation

In [ ]:
import csv
import numpy as np
from pathlib import Path
from skimage import io

# --- 1. Configuration & Paths ---
IMAGE_PATH = ".tif"
WAVELET_DIR = Path("")
CLUSTER_CSV = ""

# --- 2. Load and Downsample Original Images ---
# Load multi-stack TIFF
images = io.imread(IMAGE_PATH)
n_images = images.shape[0]

# --- 3. Load Wavelet Filtered Images ---
# Loading multi-channel wavelet data
wavelet_images = [
    io.imread(WAVELET_DIR / f"wavelet_multi_channel_id_{i}.tif")
    for i in range(n_images)
]

# --- 4. Numeric and Descriptive Labels ---
labels = []
with open(CLUSTER_CSV, mode="r", encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    next(reader)  # Skip header row
    
    for row in reader:
        cluster_id = row[-1]
        if cluster_id == '0':
            labels.append(['0', 'dox+, PhTau+'])
        elif cluster_id == '1':
            labels.append(['1', 'dox+, PhTau-'])

# Append labels for untreated cells (dox-)
while len(labels) < n_images:
    labels.append(['2', 'dox-'])

# --- 5. Extract Unique Categories ---
# Create a sorted list of human-readable category names based on cluster ID
unique_pairs = sorted(set(tuple(x) for x in labels), key=lambda x: int(x[0]))
categories = [pair[1] for pair in unique_pairs]

print(f"Loaded {n_images} images across {len(categories)} categories.")

### Spot Detection and Coordinate Mapping


In [ ]:
from skimage.feature import peak_local_max

# --- 1. Extract Wavelet Scales ---
# Extracting scales: Channel 0 is finest, Channel 1 is second finest
wavelet_scales = {
    "second": [img[:, :, 0] for img in wavelet_images],
    "third":  [img[:, :, 1] for img in wavelet_images],
    "forth":  [img[:, :, 2] for img in wavelet_images],
    "fifth":  [img[:, :, 3] for img in wavelet_images],
}

# Define the two channel groups to analyze
group_234 = [wavelet_scales["second"], wavelet_scales["third"], wavelet_scales["forth"]]
group_2345 = [wavelet_scales["second"], wavelet_scales["third"], wavelet_scales["forth"], wavelet_scales["fifth"]]

# Compute 8 combined metric scales using numpy element-wise operations
wavelet_scales["sum_234"]  = [s1 + s2 + s3 for s1, s2, s3 in zip(*group_234)]
wavelet_scales["min_234"]  = [np.minimum(s1, np.minimum(s2, s3)) for s1, s2, s3 in zip(*group_234)]
wavelet_scales["max_234"]  = [np.maximum(s1, np.maximum(s2, s3)) for s1, s2, s3 in zip(*group_234)]
wavelet_scales["mean_234"] = [tot / 3.0 for tot in wavelet_scales["sum_234"]]

wavelet_scales["sum_2345"]  = [s1 + s2 + s3 + s4 for s1, s2, s3, s4 in zip(*group_2345)]
wavelet_scales["min_2345"]  = [np.minimum(s1, np.minimum(s2, np.minimum(s3, s4))) for s1, s2, s3, s4 in zip(*group_2345)]
wavelet_scales["max_2345"]  = [np.maximum(s1, np.maximum(s2, np.maximum(s3, s4))) for s1, s2, s3, s4 in zip(*group_2345)]
wavelet_scales["mean_2345"] = [tot / 4.0 for tot in wavelet_scales["sum_2345"]]

# --- 2. Spot Detection (Peak Finding) ---
# Expanded dictionary to store detected (x, y) coordinates for all 8 combined metrics
spot_coords = {
    "sum_234":   [],
    "min_234":   [],
    "max_234":   [],
    "mean_234":  [],
    "sum_2345":  [],
    "min_2345":  [],
    "max_2345":  [],
    "mean_2345": []
}

# Parameters for peak detection
PEAK_PARAMS = {
    "min_distance": 6,
    "exclude_border": 8
}

for i in range(n_images):
    for scale_key in spot_coords.keys():
        current_img = wavelet_scales[scale_key][i]
        
        # Detect local maxima
        peaks = peak_local_max(current_img, **PEAK_PARAMS)
        
        # Convert (row, col) from skimage to (x, y) for visualisation/plotting
        coords = [(col, row) for row, col in peaks]
        spot_coords[scale_key].append(coords)

# --- 3. Verification ---
print(f"Successfully processed {n_images} images.")
for scale, data in spot_coords.items():
    print(f"{scale.capitalize()} scale: detected {len(data[0])} spots in first image.")

### Detection Visualisation


In [ ]:
import matplotlib.pyplot as plt

# --- 1. Configuration for Visualisation ---
EXAMPLE_ID = 4
VMAX_FACTOR = 3  # Contrast multiplier based on standard deviation

# Define visualisation metadata mapping the scale keys to titles and colours
plot_config = {
    "sum_234":   {"title": "Sum 234 Scale",   "colour": "#FF0055"},  # Neon Pink/Red
    "min_234":   {"title": "Min 234 Scale",   "colour": "#00FFFF"},  # Electric Cyan
    "max_234":   {"title": "Max 234 Scale",   "colour": "#FFFF00"},  # Neon Yellow
    "mean_234":  {"title": "Mean 234 Scale",  "colour": "#FF00FF"},  # Hot Magenta
    "sum_2345":  {"title": "Sum 2345 Scale",  "colour": "#00FF00"},  # Bright Lime Green
    "min_2345":  {"title": "Min 2345 Scale",  "colour": "#FF6600"},  # Blaze Orange
    "max_2345":  {"title": "Max 2345 Scale",  "colour": "#33FF99"},  # Mint Neon Green
    "mean_2345": {"title": "Mean 2345 Scale", "colour": "#CCFF00"}   # Electric Lime/Yellow
}

# --- 2. Generate Comparison Plot ---
fig, axes = plt.subplots(2, 4, figsize=(20, 10), sharex=True, sharey=True)

# Select the base image for overlay
base_img = images[EXAMPLE_ID][0]
display_vmax = base_img.std() * VMAX_FACTOR

for ax, (scale_key, meta) in zip(axes.ravel(), plot_config.items()):
    # Display the original image as background
    ax.imshow(base_img, cmap='gray', vmin=0, vmax=display_vmax*3)
    
    # Retrieve and plot detected spots
    current_spots = spot_coords[scale_key][EXAMPLE_ID]
    if current_spots:
        # Unzip coordinates (x = col, y = row)
        x_coords, y_coords = zip(*current_spots)
        ax.scatter(
            x_coords, y_coords, 
            s=1, 
            edgecolors=meta["colour"], 
            facecolors='blue', 
            linewidth=1.2,
            label=f"Count: {len(current_spots)}"
        )
    
    # Styling and Labels
    ax.set_title(meta["title"], fontsize=14, fontweight='bold')
    ax.set_axis_off()
    ax.legend(loc='upper right')

plt.suptitle(f'Spot Detection Performance Comparison (Image ID: {EXAMPLE_ID})', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])
fig.subplots_adjust(hspace=0.1)
plt.savefig(
    f"spot_detection_comparison_id_{EXAMPLE_ID}.png", 
    dpi=300, 
    bbox_inches='tight',
    transparent=False
)
plt.show()

### Delaunay Triangulation

In [ ]:
import numpy as np
from scipy.spatial import Delaunay

def compute_delaunay_triangulations(coordinate_data):
    """
    Performs Delaunay triangulation on list of spot coordinates.
    
    Args:
        coordinate_data (list): List of (x, y) coordinate tuples per image.
        
    Returns:
        list: Scipy Delaunay objects or None if triangulation is impossible.
    """
    results = []
    for spot_coords in coordinate_data:
        # Delaunay triangulation requires a minimum of 4 non-collinear points in 2D 
        # to avoid most geometric errors, though 3 is the theoretical minimum.
        if len(spot_coords) >= 3:
            points = np.array(spot_coords)
            try:
                tri = Delaunay(points)
                results.append(tri)
            except Exception as e:
                # Logs errors such as collinear points or precision issues
                print(f"Warning: Triangulation failed for an image: {e}")
                results.append(None)
        else:
            results.append(None)
    return results

# --- 1. Compute Triangulations ---
# Dictionary to store the Delaunay objects for each detection scale
delaunay_results = {}

for scale_key, coords in spot_coords.items():
    delaunay_results[scale_key] = compute_delaunay_triangulations(coords)
    
    # Calculate success rate for this scale
    valid_count = sum(1 for x in delaunay_results[scale_key] if x is not None)
    print(f"Processed '{scale_key}': {valid_count}/{n_images} images successfully triangulated.")

# --- 2. Data Inspection Example ---
# Inspecting the first image from the 'finest' detection scale
example_tri = delaunay_results["min_234"][0]

if example_tri:
    print(f"\nExample Metadata (Scale: 'min_234', Image ID: 0):")
    print(f"  - Number of Triangles (Simplices): {len(example_tri.simplices)}")
    print(f"  - Number of Input Vertices: {len(example_tri.points)}")
else:
    print("\nExample image does not contain enough spots for triangulation.")

### Triangulation Visualisation

### Data Export

In [ ]:
import pickle
from pathlib import Path

# --- 1. Define Output Directory ---
# Using Path ensures compatibility across Windows, macOS, and Linux
output_dir = Path("")
output_dir.mkdir(exist_ok=False)

# --- 2. Serialise Results for Downstream Analysis ---
# Data to save: spot coordinates and triangulation objects
export_files = {
    "spot_coordinates.pkl": spot_coords,
    "delaunay_results.pkl": delaunay_results
}

for filename, data in export_files.items():
    save_path = output_dir / filename
    with open(save_path, "wb") as f:
        pickle.dump(data, f)
    
    print(f"Successfully exported: {save_path} ({len(data)} detection scales)")